# Llama 3.1 8B Instruct on Colab — Milestones 1 & 2

Runs the project's own scripts on a GPU. Nothing here is Colab-specific logic;
it installs deps, gets the code onto the VM, and calls `scripts/`.

**Before you start:** Runtime → Change runtime type → **T4 GPU**.

### The T4 constraint, up front

A free-tier T4 has ~15 GiB of VRAM. Llama 3.1 8B in fp16 is ~15.0 GiB of
weights *alone*, so full precision does not fit — there is no room left for the
KV cache. This notebook therefore uses **4-bit NF4** (~5.9 GiB), which fits
comfortably.

4-bit quantisation does perturb activations. Refusal directions and steering
vectors remain recoverable, but for final publishable numbers use a ≥24 GiB card
(L4, 3090, 4090, A100) in fp16. Treat T4 results as development-grade.

## 1. Check the GPU

In [ ]:
!nvidia-smi

In [ ]:
import torch
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"{p.name}: {p.total_memory/1024**3:.1f} GiB, bf16={torch.cuda.is_bf16_supported()}")
else:
    print("NO GPU -- Runtime > Change runtime type > T4 GPU, then rerun.")

## 2. Get the code onto the VM

**Option A — from GitHub (recommended).** Push the project first, then set
`REPO_URL` below. For a private repo use
`https://<token>@github.com/<you>/<repo>.git`.

**Option B — upload a zip.** Zip the project locally (exclude `.venv/`), then run
the upload cell and skip the clone.

In [ ]:
REPO_URL = ""  # e.g. "https://github.com/<you>/major_project.git"

if REPO_URL:
    !git clone -q $REPO_URL /content/major_project
    print("Cloned.")
else:
    print("No REPO_URL set -- use the upload cell below instead.")

In [ ]:
# Option B: upload a zip of the project. Skip this cell if you cloned above.
import pathlib, zipfile

if not pathlib.Path("/content/major_project").exists():
    from google.colab import files
    uploaded = files.upload()          # select your project .zip
    name = next(iter(uploaded))
    with zipfile.ZipFile(name) as z:
        z.extractall("/content/major_project_upload")
    # Handle both zip-of-folder and zip-of-contents layouts.
    root = pathlib.Path("/content/major_project_upload")
    inner = next((p for p in root.rglob("requirements.txt")), None)
    assert inner, "requirements.txt not found in the uploaded zip"
    pathlib.Path("/content/major_project").symlink_to(inner.parent)
    print("Extracted to", inner.parent)
else:
    print("/content/major_project already exists -- nothing to upload.")

In [ ]:
%cd /content/major_project

In [ ]:
!ls -la

## 3. Install dependencies

Colab ships torch with CUDA already, so we do **not** reinstall it — that would
pull a multi-GB wheel and can break the CUDA build. We add only what is missing.

In [ ]:
!pip install -q -U transformers accelerate huggingface_hub safetensors

In [ ]:
!pip install -q bitsandbytes

## 4. Hugging Face authentication

`meta-llama/Llama-3.1-8B-Instruct` is gated. You must have accepted the licence
at <https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct> with the same
account that owns this token.

Best practice: store the token in Colab's **Secrets** panel (🔑 in the sidebar)
as `HF_TOKEN` rather than pasting it into a cell — notebook output gets shared.

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("Token loaded from Colab Secrets.")
except Exception as e:
    print("Secrets unavailable (%s); falling back to interactive login." % type(e).__name__)
    from huggingface_hub import notebook_login
    notebook_login()

## 5. Environment report — confirms access and sizing

In [ ]:
!python scripts/00_check_env.py

## 6. Milestone 1 — load and generate

The first run downloads ~16 GiB of weights: roughly 5–10 minutes. Colab's disk is
ephemeral, so this repeats on every fresh session.

In [ ]:
!python scripts/01_load_and_generate.py --load-in-4bit --max-new-tokens 256

## 7. Milestone 2 — capture hidden activations

`resid_post` is the residual stream. Add `attn_out`, `mlp_out`, and `resid_mid`
to decompose each layer's contribution.

Sizing: 32 layers × 4096 dims × 4 bytes ≈ **512 KiB per prompt per site** at the
last-token position. A 1,000-prompt sweep across all four sites is ~2 GiB.

In [ ]:
!python scripts/02_capture_activations.py --load-in-4bit --sites resid_post attn_out mlp_out resid_mid --batch-size 8 --run-name llama31_8b_smoke


In [ ]:
# Inspect what was captured.
import sys
sys.path.insert(0, "/content/major_project")
from src.activations import load_activation_set

acts = load_activation_set("data/activations/llama31_8b_smoke")
print(acts.summary())
print()
print("provenance:", acts.provenance)

layer15 = acts.layer("resid_post", 15)     # [n_prompts, 4096]
print()
print("layer 15 residual stream:", tuple(layer15.shape))
print("norms per prompt:", [round(x, 2) for x in layer15.norm(dim=-1).tolist()])

## 8. Your own prompt set

One user prompt per line; `#` comments and blank lines are ignored. The script
applies the Llama 3.1 chat template to each one before capture — the step that
keeps activations on-distribution.

In [ ]:
%%writefile my_prompts.txt
# One prompt per line.
What is the capital of France?
Explain photosynthesis in two sentences.
How do I bake sourdough bread?

In [ ]:
!python scripts/02_capture_activations.py --load-in-4bit --prompts my_prompts.txt --sites resid_post --run-name my_run


## 9. Persist results to Drive

Colab wipes local disk when the session ends. Activations are the expensive
artefact — copy them out before you disconnect.

Caching the model weights to Drive is usually **not** worth it: reading 16 GiB
back over the Drive mount is often slower than re-downloading from HF.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!mkdir -p /content/drive/MyDrive/major_project
!cp -r data/activations /content/drive/MyDrive/major_project/
!cp -r outputs/generations /content/drive/MyDrive/major_project/
!du -sh /content/drive/MyDrive/major_project/*

## Notes

- **Session limits.** Free Colab disconnects after ~12h, and sooner when idle.
  Long activation sweeps should checkpoint: split the prompt file and use a
  distinct `--run-name` per chunk.
- **Out of memory.** Lower `--batch-size` first — it is a pure performance knob,
  and `tests/test_activations.py` asserts results do not depend on it.
- **`--token-position all`** stores every position, not just the last. That is
  `seq_len` times larger; use it only on a small prompt subset.
- **fp16 instead of 4-bit** needs ≥24 GiB. On such a card, drop `--load-in-4bit`
  and the loader picks fp16 (or bf16 on Ampere and newer) automatically.